# VectorDrugMatch - final Colab run (PrimeKG, both benchmark variants)

Run the cells **top to bottom**. Everything is driven by `scripts/run_full_pipeline.py`; the code is **pinned to commit `c3deddce78`** so the run is reproducible.
Before running: that commit must be pushed to GitHub (`origin/mukul`), and your private files must be in Drive (Step 3).
Training happens only here; local work produced the data layer, audits and exporters. Read `Docs/DATA_AND_PIPELINE_READINESS.md` first.

## Step 1 - GPU
Runtime > Change runtime type > T4 GPU. Expect `GPU available: True`.

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
!nvidia-smi | head -15

## Step 2 - Drive, code (pinned commit) and libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/VDM_Data /content/drive/MyDrive/VDM_Outputs
%cd /content
!rm -rf VectorDrugMatch
!git clone https://github.com/MUKUL-PRASAD-SIGH/VectorDrugMatch
%cd VectorDrugMatch
PIN = 'c3deddce78116e2d8d1fc945e054d1dfa88e82ac'
!git checkout $PIN
!git log -1 --oneline

In [ ]:
!pip install -q rdkit mlflow optuna shap xgboost lightgbm loguru pyyaml torch-geometric tabulate scikit-learn lxml matplotlib
import torch
v = torch.__version__.split('+')[0]; c = (torch.version.cuda or '').replace('.', '')
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-{v}+cu{c}.html
import torch_geometric, torch_cluster
print('ok:', torch.__version__, torch_geometric.__version__, 'torch_cluster available')

## Step 3 - Put your private/manual inputs in place (from Drive)
Copy these into Drive once, under `MyDrive/VDM_Data/` : the folder `downloads/` (it holds `open_targets/`, `hai_price_surveys/`, `ihme_gbd/` exactly as locally),
and optionally `who_eml.csv`. Missing items are fine: the matching components are reported as **missing**, never as zero.

In [ ]:
import os, shutil
D = '/content/drive/MyDrive/VDM_Data'
os.makedirs('data/external', exist_ok=True)
if os.path.isdir(f'{D}/downloads'):
    shutil.copytree(f'{D}/downloads', 'data/external/downloads', dirs_exist_ok=True); print('copied downloads/')
else:
    print('no downloads/ in Drive: Open Targets, HAI and IHME steps will be skipped (optional)')
if os.path.exists(f'{D}/who_eml.csv'):
    shutil.copy(f'{D}/who_eml.csv', 'data/external/who_eml.csv'); print('copied who_eml.csv')
if os.path.exists(f'{D}/drugbank_smiles_cache.json'):
    shutil.copy(f'{D}/drugbank_smiles_cache.json', 'data/external/drugbank_smiles_cache.json'); print('restored SMILES cache')

## Step 4 - Prepare: download PrimeKG + MONDO, build tables, save split, run the safety check, build the structure-covered variant
Stops at the first failure. A validation `[FAIL]` means: do not train.

In [ ]:
!python scripts/run_full_pipeline.py --stage prepare
!cp data/external/drugbank_smiles_cache.json /content/drive/MyDrive/VDM_Data/ 2>/dev/null || true
!cp -r data/processed/primekg data/processed/primekg_structcov /content/drive/MyDrive/VDM_Data/ 2>/dev/null || true

## Step 5 - External evidence and the equity table (Open Targets, HAI, IHME, EML are optional inputs)

In [ ]:
!python scripts/run_full_pipeline.py --stage external

## Step 6 - Smoke test (do this before the long run)
Checks `torch_cluster`, Node2Vec for one epoch, five epochs for the five GNNs, one baseline. Look at runtime and GPU memory.

In [ ]:
%env MLFLOW_TRACKING_URI=sqlite:///mlflow.db
!python scripts/run_full_pipeline.py --stage smoke

## Step 7 - Full benchmark: baselines, Node2Vec, five GNNs, cold-start ablation, then evaluation and the paper results package
Baselines run one model per call. If Colab disconnects, re-run Steps 1-3 and this step: finished prediction files are rewritten, nothing else is affected.

In [ ]:
!python scripts/run_full_pipeline.py --stage train --variant full
!python scripts/run_full_pipeline.py --stage report --variant full
!mkdir -p /content/drive/MyDrive/VDM_Outputs && cp -r outputs/primekg /content/drive/MyDrive/VDM_Outputs/primekg_full

## Step 8 - Structure-covered variant (same drug partition, drugs with a usable fingerprint only)

In [ ]:
!python scripts/run_full_pipeline.py --stage train --variant structcov
!python scripts/run_full_pipeline.py --stage report --variant structcov
!cp -r outputs/primekg_structcov /content/drive/MyDrive/VDM_Outputs/primekg_structcov

## Step 9 - Freeze, package and download
Writes the release manifest (checksums, versions, commit), zips the results package, saves a copy to Drive and downloads it.

In [ ]:
!python scripts/freeze_release.py --tag vdm_v1
!python scripts/make_readiness_report.py --skip-validators
!zip -rq vdm_results.zip outputs/primekg/paper_package outputs/primekg_structcov/paper_package outputs/primekg/*.csv outputs/primekg_structcov/*.csv outputs/primekg/models/*.json outputs/primekg_structcov/models/*.json outputs/pipeline_log.jsonl Docs/release data/processed/equity data/processed/ntd data/processed/open_targets data/processed/hai
!cp vdm_results.zip /content/drive/MyDrive/VDM_Outputs/
from google.colab import files
files.download('vdm_results.zip')

## Reading the results honestly
Report the `all` rows first; show `with_structure` / `no_structure` next to them; NTD rows have ~15 test positives (exploratory). Use only `coldstart_*` baseline numbers (the printed 5-fold CV pools drugs and is easier). The legacy `train_benchmark.py` Node2Vec and `run_ablations.py` are not drug-disjoint: never report them.